In [ ]:
#REST API & UGET 1m DEM tiles URL 

import geopandas as gpd
import requests
from shapely.geometry import box

# ===============================
# USER SETTINGS
# ===============================
shapefile_path = r"C:\Users\KyleSteen\Documents\I90_ROW_Buf_1500ft\I90_ROW_Buf_1500ft.shp"
save_urls_file = r"C:\Users\KyleSteen\Documents\dem_urls.txt"

# ===============================
# 1. Load AOI and get bounding box in WGS84
# ===============================
gdf_aoi = gpd.read_file(shapefile_path)
gdf_aoi = gdf_aoi.to_crs(epsg=4326)
polygon = gdf_aoi.geometry.union_all()
minx, miny, maxx, maxy = polygon.bounds
print(f"AOI bounding box (WGS84): {minx}, {miny}, {maxx}, {maxy}")

# ===============================
# 2. Query USGS 3DEP Tile Index REST API
# ===============================
# This is the 1m DEM dataset endpoint
api_url = "https://tnmaccess.nationalmap.gov/api/v1/products"

params = {
    "datasets": "3DEP 1-meter DEM",
    "bbox": f"{minx},{miny},{maxx},{maxy}",
    "max": 1000,
    "format": "json"
}

response = requests.get(api_url, params=params)
response.raise_for_status()
data = response.json()

# ===============================
# 3. Extract download URLs
# ===============================
urls = []
for item in data.get("items", []):
    # Usually download links are in 'urls'
    for u in item.get("urls", []):
        url = u.get("url")
        if url:
            urls.append(url)

print(f"Found {len(urls)} 1m DEM tiles around AOI.")

# ===============================
# 4. Save URLs to file for uGet
# ===============================
with open(save_urls_file, "w") as f:
    for url in urls:
        f.write(url + "\n")

print(f"URLs saved to {save_urls_file}")

